In [ ]:
from transformers.retrieval_rag import RagRetriever


retriever = RagRetriever.from_pretrained("models/rag-sequence-nq")

In [2]:
retriever.init_retrieval()

In [ ]:
from transformers.tokenization_rag import RagTokenizer

from securerag.eval import RagSequenceForGeneration

rag_seq = RagSequenceForGeneration.from_pretrained(
    "models/rag-sequence-nq", retriever=retriever
)

/home/huahua/miniconda3/envs/SecureRAG/lib/python3.8/site-packages/transformers/modeling_utils.py:927: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(

In [ ]:
tokenizer = RagTokenizer.from_pretrained("models/rag-sequence-nq")
question_encoder = rag_seq.question_encoder

In [5]:
from securerag import data
from securerag.config import Config

cfg = Config()
debug_data = data.load(path="data/open_domain_data/NQ/debug.json", cfg=cfg)

In [ ]:
question = debug_data[0]["question"]
question

'who sings does he love me with reba'

In [7]:
input_dict = tokenizer.prepare_seq2seq_batch(question, return_tensors="pt")
input_ids = input_dict["input_ids"]
input_mask = input_dict["attention_mask"]

In [ ]:
question_enc_outputs = question_encoder(
    input_ids, attention_mask=input_mask, return_dict=True
)
question_encoder_last_hidden_state = question_enc_outputs[
    0
]  # hidden states of question encoder

In [ ]:
import torch


retriever_outputs = retriever(
    input_ids,
    question_encoder_last_hidden_state.cpu().detach().to(torch.float32).numpy(),
    prefix="",
    n_docs=100,
    return_tensors="pt",
)

/home/huahua/miniconda3/envs/SecureRAG/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:555: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  tensor = as_tensor(value)


In [10]:
doc_ids = retriever_outputs["doc_ids"]

In [11]:
doc_ids

tensor([[ 7624371,  7624379, 11828866,  2222001,  1670046, 11828867, 11828868,
          9968579,  7624375, 10642285, 11828869,  2222024,  9968585, 19442163,
          9968578,  1670064,  3297761,  9968583, 14605379,  4103910,  9997249,
         10795194, 19463969,  8022021,  7703370, 11782383,  3297759,  9968581,
          8815881,  9968580,  1670047,  8870096, 11828872,  9749742,  8870095,
          1670058, 11782382,  9968586,  7800843, 14015676, 14015678,  8279934,
          2905617, 18673033,  7831437,  9078577,  1670028,  1361292, 15987613,
         10752935, 11782381, 13029767,  8279936, 13029759,  1670067, 16819913,
         17455628,  4687099,  1745235,  4460667, 13008046, 13008047, 10752938,
          8065472,  4510642, 17974520, 16819914,  1361345, 13129764, 11179141,
         19005927, 15758321, 19463971,  8531631,  2905619, 18351764, 10020179,
         15438843, 13108970,  3888444,  9992262, 10880283,  1312061,  8905114,
          6000465,  6104494,  1910408, 11506430, 201

In [102]:
embs = retriever_outputs["retrieved_doc_embeds"]
torch.mm(question_encoder_last_hidden_state, embs.transpose(1, 0))

tensor([[89.2454, 86.4275, 85.9853, 83.9592, 83.8910, 83.6696, 83.8792, 83.1254,
         82.8279, 82.5546, 82.5527, 82.5270, 82.4865, 82.2560, 82.3904, 82.1815,
         82.2051, 82.1550, 81.9790, 82.0121, 81.5845, 81.7333, 81.7528, 81.6574,
         81.5073, 81.5417, 81.4394, 81.3909, 81.5781, 81.4680, 81.1557, 81.4765,
         81.3838, 81.3385, 81.2591, 81.2744, 81.2562, 81.0573, 80.9110, 81.0838,
         81.0837, 80.9437, 81.0335, 80.7945, 81.0266, 80.7924, 80.9851, 80.7314,
         81.1040, 81.1189, 80.9092, 80.8366, 80.7953, 80.6623, 80.7172, 80.6658,
         80.6112, 80.6577, 80.7664, 80.6454, 80.8032, 80.8032, 80.6618, 80.5938,
         80.5965, 80.6861, 80.6672, 80.5376, 80.7696, 80.6052, 80.8445, 80.5133,
         80.6992, 80.4017, 80.6960, 80.5745, 80.7248, 80.5424, 80.5418, 80.3626,
         80.3555, 80.5803, 80.6264, 80.4146, 80.5419, 80.2538, 80.3019, 80.3577,
         80.2871, 80.3948, 80.3564, 80.2389, 80.1809, 80.3218, 80.1371, 80.0288,
         80.1871, 80.3752, 8

In [218]:
def get_scores(question):
    input_dict = tokenizer.prepare_seq2seq_batch(question, return_tensors="pt")
    input_ids = input_dict["input_ids"]
    input_mask = input_dict["attention_mask"]
    question_enc_outputs = question_encoder(
        input_ids, attention_mask=input_mask, return_dict=True
    )
    question_encoder_last_hidden_state = question_enc_outputs[
        0
    ]  # hidden states of question encoder
    (embs, doc_ids, doc_dicts) = retriever_outputs = retriever.retrieve(
        question_encoder_last_hidden_state.cpu().detach().to(torch.float32).numpy(),
        n_docs=100,
    )
    doc_ids = doc_ids[0]
    doc_dict = doc_dicts[0]
    embs = torch.Tensor(embs)
    embs = embs.squeeze(0)
    scores = torch.mm(question_encoder_last_hidden_state, embs.transpose(1, 0))[0].tolist()
    titles = doc_dict["title"]
    texts = doc_dict["text"]
    ret = [
        {
            "id": int(doc_id),
            "title": titles[i],
            "text": texts[i],
            "score": scores[i]
        }
        for i, doc_id in enumerate(doc_ids)
    ]
    return ret

In [219]:
tmp = get_scores(question)
print(tmp[0])

{'id': 7624371, 'title': 'Linda Davis', 'text': 'Linda Davis Linda Kaye Davis (born November 26, 1962) is an American country music singer. Before beginning a career as a solo artist, she had three minor country singles in the charts as one half of the duo Skip & Linda. In her solo career, Davis has recorded five studio albums for major record labels and more than 15 singles. Her highest chart entry is "Does He Love You", her 1993 duet with Reba McEntire, which reached number one on the "Billboard" country charts and won both singers the Grammy for Best Country Vocal Collaboration. Her highest solo chart position', 'score': 89.24541473388672}


In [ ]:
import json

from tqdm import tqdm

output_path = "data/open_domain_data/NQ/dev_with_scores.json"
input_path = "data/open_domain_data/NQ/dev.json"

examples = []
with open(input_path, "r") as fin, open(output_path, "w") as fout:
    json_data = json.load(fin)
    for k, example in tqdm(enumerate(json_data)):
        question = example["question"]
        example["ctxs"] = get_scores(question)
        examples.append(example)
    json.dump(examples, fout)

1652it [02:16, 12.54it/s]